In [ ]:
# pip install MAPIE


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, platform, importlib.metadata, uuid
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from scipy.signal import detrend
from sklearn.base import BaseEstimator, ClassifierMixin
try:
    from IPython.display import display
except ImportError:

    def display(value):
        print(value.to_string() if isinstance(value, pd.DataFrame) else value)


In [ ]:
ROOT = Path(os.environ.get('PINPAD_DATA_ROOT', '/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate (Copy)')).expanduser()
OUTPUT = Path(os.environ.get('PINPAD_CP_OUTPUT', str(Path.cwd() / 'CP_Results'))).expanduser()
CAPTURES_CSV = ROOT / 'cp_capture_manifest.csv'
MODELS = ('DL', 'RF')
RUN_IDS = ('run1', 'run2', 'run3', 'run4', 'run5')
N_BUTTONS, N_PER_BUTTON, RESPONSE_LENGTH = (12, 200, 3624)
CLASS_ORDER = [f'B{i}' for i in range(N_BUTTONS)]
ALPHA = {'DL': 0.07, 'RF': 0.05}
TAU, MAX_CAPTURES = (1.0, 3)
SCORE = 'aps'
INCLUDE_LAST_LABEL = True
CP_SEED = 42
SELECTION_MODE = 'validation'
TIE_RULE = 'lowest_class_index'
KEEP_PLACEMENT_BLOCKS_TOGETHER = False
RISK_LEVELS = np.arange(5, 31, dtype=float) / 100
THRESHOLDS = np.arange(0.5, 1.001, 0.1).round(2)
CAPTURE_BUDGETS = (1, 2, 3)
TEST = {'CC': [(c, 'R1') for c in range(16, 26)], 'CR': [(c, 'R3') for c in range(1, 6)], 'CCR': [(c, 'R3') for c in range(26, 36)]}
DEVELOPMENT = {'Source': [(c, 'R1') for c in range(1, 6)], 'CC': [(c, 'R1') for c in range(6, 11)], 'CR': [(c, 'R2') for c in range(1, 6)], 'CCR': [(c, 'R2') for c in range(11, 16)]}
EXPECTED_COUNTS = {'CC': 24000, 'CR': 12000, 'CCR': 24000}
PREPROCESSING = {'representation': 'magnitude', 'drift_removal': 'linear', 'normalization': 'per_response_zscore', 'std_epsilon': 1e-12, 'signal_length': RESPONSE_LENGTH}
if SCORE not in ('aps', 'lac') or SELECTION_MODE not in ('validation', 'fixed'):
    raise ValueError('Invalid score or selection mode')


In [ ]:
def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def array_hash(x):
    array = np.ascontiguousarray(x)
    return hashlib.sha256(array.view(np.uint8)).hexdigest()


In [ ]:
def resolve_collections(groups):
    expected = set((k for keys in groups.values() for k in keys))
    mapping = {}
    manifest = ROOT / 'collection_paths.csv'
    if manifest.is_file():
        frame = pd.read_csv(manifest, dtype=str, keep_default_na=False)
        if not {'card', 'reader', 'directory'}.issubset(frame.columns):
            raise ValueError('collection_paths.csv requires card, reader, directory')
        for row in frame.itertuples(index=False):
            key = (int(row.card), row.reader.strip().upper())
            if key in mapping or not row.directory.strip():
                raise ValueError('Duplicate or empty collection mapping')
            directory = Path(row.directory).expanduser()
            mapping[key] = (directory if directory.is_absolute() else ROOT / directory).resolve()
    for card, reader in sorted(expected - set(mapping)):
        domain = next((d for d, keys in groups.items() if (card, reader) in keys))
        candidates = [ROOT / reader / f'card{card}', ROOT / reader.lower() / f'card{card}', ROOT / f'card{card}' / reader, ROOT / f'card{card}' / reader.lower(), ROOT / domain / reader / f'card{card}', ROOT / domain / reader.lower() / f'card{card}']
        if domain in ('Source', 'CC'):
            candidates.append(ROOT / domain / f'card{card}')
        if domain == 'CR':
            candidates.append(ROOT / 'CR_' / reader / f'card{card}')
        hits = {p.resolve() for p in candidates if p.is_dir()}
        if len(hits) != 1:
            raise FileNotFoundError(f'Provide collection_paths.csv for card {card}, {reader}: {sorted(hits)}')
        mapping[card, reader] = hits.pop()
    if any((not mapping[k].is_dir() for k in expected)) or len({mapping[k] for k in expected}) != len(expected):
        raise ValueError('Missing or aliased collection directories')
    return {k: mapping[k] for k in expected}

def load_capture_manifest():
    f = pd.read_csv(CAPTURES_CSV, dtype=str, keep_default_na=False)
    required = {'domain', 'card', 'reader', 'button', 'sample_index', 'partition', 'episode_id', 'capture_index', 'session_id', 'capture_time'}
    if not required <= set(f):
        raise ValueError(f'Missing capture fields: {sorted(required - set(f))}')
    for col in ('card', 'button', 'sample_index'):
        f[col] = pd.to_numeric(f[col], errors='raise').astype(int)
    f['reader'] = f.reader.str.upper().str.strip()
    if set(f.domain) != set(TEST):
        raise ValueError('CC, CR, CCR required')
    if set(f.partition) != {'calibration', 'validation', 'final'}:
        raise ValueError('Invalid partition names')
    if not f.button.between(0, 11).all() or not f.sample_index.between(0, 199).all():
        raise ValueError('Invalid button/sample index')
    if f.duplicated(['card', 'reader', 'button', 'sample_index']).any():
        raise ValueError('Duplicate capture')
    for domain, g in f.groupby('domain'):
        if len(g) != EXPECTED_COUNTS[domain] or set(zip(g.card, g.reader)) != set(TEST[domain]):
            raise ValueError(f'{domain}: collection mismatch')
        if not g.groupby(['card', 'reader', 'button']).size().eq(200).all():
            raise ValueError('Expected 200 captures per button')
        for role, fraction in [('calibration', 0.1), ('validation', 0.1), ('final', 0.8)]:
            if len(g[g.partition == role]) != round(len(g) * fraction):
                raise ValueError(f'{domain}: {role} response count is not {fraction:.0%}')
            if set(g.loc[g.partition == role, 'button']) != set(range(12)):
                raise ValueError(f'{domain}/{role}: missing button classes')
        if KEEP_PLACEMENT_BLOCKS_TOGETHER:
            blocks = g.assign(block=g.sample_index // 50)
            if blocks.groupby(['card', 'reader', 'button', 'block']).partition.nunique().gt(1).any():
                raise ValueError('Placement block crosses partitions')
    linked = f[f.episode_id.ne('')]
    if linked.groupby(['domain', 'episode_id']).partition.nunique().gt(1).any():
        raise ValueError('Episode crosses partitions')
    episodes = f[f.partition != 'calibration']
    if episodes[['episode_id', 'session_id', 'capture_time', 'capture_index']].eq('').any().any():
        raise ValueError('Missing episode metadata')
    for key, g in episodes.groupby(['domain', 'episode_id']):
        if any((g[col].nunique() != 1 for col in ('partition', 'card', 'reader', 'button', 'session_id'))):
            raise ValueError(f'Mixed provenance/partition within episode {key}')
        g = g.assign(capture_index=pd.to_numeric(g.capture_index, errors='raise').astype(int)).sort_values('capture_index')
        if len(g) != MAX_CAPTURES or g.capture_index.tolist() != list(range(1, MAX_CAPTURES + 1)):
            raise ValueError(f'{key}: invalid capture order')
        times = pd.to_datetime(g.capture_time, utc=True, errors='raise')
        if times.isna().any() or (times.diff().iloc[1:] <= pd.Timedelta(0)).any():
            raise ValueError(f'{key}: timestamps must strictly increase')
    return f.reset_index(drop=True)


In [ ]:
def load_test_responses(manifest):
    mapping = resolve_collections(TEST)
    responses = {}
    for domain in TEST:
        rows = manifest[manifest.domain == domain].copy().reset_index(drop=True)
        values = [None] * len(rows)
        for (card, reader, button), g in rows.groupby(['card', 'reader', 'button'], sort=False):
            folder = mapping[int(card), reader]
            hits = [p for p in (folder / f'button{button}.npy', folder / f'Button{button}.npy') if p.is_file()]
            if len(hits) != 1:
                raise FileNotFoundError(f'Expected one button{button}.npy in {folder}')
            raw = np.load(hits[0], allow_pickle=False)
            if raw.shape != (200, RESPONSE_LENGTH) or not np.iscomplexobj(raw) or (not np.isfinite(raw).all()):
                raise ValueError(f'Invalid response array: {hits[0]}')
            for index, row in g.iterrows():
                values[index] = raw[int(row.sample_index)]
        x = np.stack(values)
        rows['raw_sha256'] = [array_hash(v) for v in x]
        rows['response_sha256'] = [array_hash(v) for v in preprocess(x, PREPROCESSING)]
        if rows.groupby('response_sha256').partition.nunique().gt(1).any():
            raise ValueError(f'{domain}: waveform crosses partitions')
        measured = rows[rows.partition != 'calibration']
        if measured.response_sha256.duplicated().any():
            raise ValueError(f'{domain}: duplicate waveform')
        responses[domain] = (x, rows)
    combined = pd.concat([rows for x, rows in responses.values()], ignore_index=True)
    if combined.groupby('response_sha256').partition.nunique().gt(1).any():
        raise ValueError('Waveform crosses partitions')
    if combined[combined.partition != 'calibration'].response_sha256.duplicated().any():
        raise ValueError('Repeated evaluation waveform')
    return responses

def audit_development(samples, destination):
    mapping = resolve_collections(DEVELOPMENT)
    hashes, records = (set(), [])
    for domain, keys in DEVELOPMENT.items():
        for card, reader in keys:
            for button in range(N_BUTTONS):
                folder = mapping[card, reader]
                hits = [p for p in (folder / f'button{button}.npy', folder / f'Button{button}.npy') if p.is_file()]
                if len(hits) != 1:
                    raise FileNotFoundError(f'Expected button{button}.npy in {folder}')
                raw = np.load(hits[0], allow_pickle=False)
                if raw.shape != (N_PER_BUTTON, RESPONSE_LENGTH):
                    raise ValueError(f'Invalid development array: {hits[0]}')
                for index, row in enumerate(preprocess(raw, PREPROCESSING)):
                    h = array_hash(row)
                    hashes.add(h)
                    records.append({'domain': domain, 'card': card, 'reader': reader, 'button': button, 'sample_index': index, 'response_sha256': h})
    pd.DataFrame(records).to_csv(destination / 'development_input_hashes.csv', index=False)
    for domain, (_, rows) in samples.items():
        overlap = rows[rows.response_sha256.isin(hashes)]
        if not overlap.empty:
            overlap.to_csv(destination / f'{domain}_overlapping_inputs.csv', index=False)
            raise ValueError(f'{domain}: development and CP responses overlap')
    return hashes


In [ ]:
def preprocess(raw, prep):
    required = (prep.get('representation'), prep.get('drift_removal'), prep.get('normalization'))
    if required != ('magnitude', 'linear', 'per_response_zscore'):
        raise ValueError('Saved preprocessing differs from the selected training pipeline')
    x = np.asarray(raw)
    if x.ndim != 2 or x.shape[1] != prep['signal_length'] or (not np.iscomplexobj(x)) or (not np.isfinite(x).all()):
        raise ValueError('Expected finite complex response rows')
    x = detrend(np.abs(x).astype(np.float64), axis=1, type='linear')
    std = x.std(axis=1, keepdims=True)
    if np.any(std <= prep['std_epsilon']):
        raise ValueError('Response has negligible variation after drift removal')
    return ((x - x.mean(axis=1, keepdims=True)) / std).astype(np.float32)

def softmax_logits(logits):
    z = np.asarray(logits, dtype=np.float64)
    if z.ndim != 2 or z.shape[1] != N_BUTTONS or (not np.isfinite(z).all()):
        raise ValueError('Expected 12 finite logits per response')
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return checked_probabilities(e / e.sum(axis=1, keepdims=True))

def checked_probabilities(p):
    p = np.asarray(p, dtype=np.float64)
    if p.ndim != 2 or p.shape[1] != 12 or (not np.isfinite(p).all()) or np.any(p < 0) or np.any(p > 1):
        raise ValueError('Invalid class probabilities')
    if not np.allclose(p.sum(axis=1), 1.0, atol=1e-06):
        raise ValueError('Probabilities do not sum to one')
    return p / p.sum(axis=1, keepdims=True)


In [ ]:
def model_files(name, run_id):
    if name == 'DL':
        if DL_RUN_ROOT is None:
            raise ValueError('Set DL_RUN_ROOT to the completed DL full_run directory containing run1 through run5')
        folder = Path(DL_RUN_ROOT) / run_id
        paths = {'model': folder / 'adapted' / 'adapted_model.keras', 'preprocessing': folder / 'normalization.json'}
    elif name == 'RF':
        if RF_RUN_ROOT is None:
            raise ValueError('Set RF_RUN_ROOT to the completed RF run directory containing run_01 through run_05')
        number = RUN_IDS.index(run_id) + 1
        paths = {'model': Path(RF_RUN_ROOT) / f'run_{number:02d}' / 'RF_models.joblib'}
    else:
        raise ValueError('Unknown model')
    for path in paths.values():
        if not path.is_file():
            raise FileNotFoundError(path)
    return paths

def load_predictor(name, run_id):
    paths = model_files(name, run_id)
    metadata = {'model_path': str(paths['model'].resolve()), 'model_sha256': file_sha256(paths['model'])}
    if name == 'DL':
        import tensorflow as tf
        model = tf.keras.models.load_model(paths['model'], compile=False)
        model.trainable = False
        prep = json.loads(paths['preprocessing'].read_text())
        if prep.get('class_order') != CLASS_ORDER or tuple(model.input_shape) != (None, RESPONSE_LENGTH, 1) or tuple(model.output_shape) != (None, N_BUTTONS):
            raise ValueError('DL model shape or class configuration differs')
        if tf.keras.activations.serialize(model.get_layer('button_classifier').get_layer('button_logits').activation) != 'linear':
            raise ValueError('Expected linear logits')

        def predict(raw):
            x = preprocess(raw, prep)[..., None]
            return softmax_logits(model.predict(x, batch_size=256, verbose=0))
        metadata.update(preprocessing_path=str(paths['preprocessing'].resolve()), preprocessing_sha256=file_sha256(paths['preprocessing']))
    else:
        bundle = joblib.load(paths['model'])
        prep = bundle['preprocessing']
        pca, model = (bundle['pca'], bundle['rf'])
        if bundle['classes'] != CLASS_ORDER or not np.array_equal(model.classes_, np.arange(N_BUTTONS)):
            raise ValueError('RF class order differs')

        def predict(raw):
            return checked_probabilities(model.predict_proba(pca.transform(preprocess(raw, prep))))
    if any((prep.get(k) != v for k, v in PREPROCESSING.items())):
        raise ValueError('Saved preprocessing does not match the input audit configuration')
    metadata['preprocessing'] = prep
    return (predict, metadata)


In [ ]:
class FrozenProbabilities(ClassifierMixin, BaseEstimator):

    def __init__(self):
        self.classes_ = np.arange(12)
        self.n_features_in_ = 12

    def __sklearn_is_fitted__(self):
        return True

    def fit(self, X, y):
        raise RuntimeError('Model is frozen')

    def predict_proba(self, X):
        return checked_probabilities(X)

    def predict(self, X):
        return np.asarray(X).argmax(axis=1)

def calibrate(probabilities, labels, alpha):
    from mapie.classification import SplitConformalClassifier
    cp = SplitConformalClassifier(estimator=FrozenProbabilities(), confidence_level=1 - float(alpha), conformity_score=SCORE, prefit=True, random_state=CP_SEED)
    cp.conformalize(probabilities, labels)
    return cp

def get_sets(cp, probabilities):
    params = {'include_last_label': INCLUDE_LAST_LABEL} if SCORE == 'aps' else None
    _, sets = cp.predict_set(probabilities, conformity_score_params=params)
    sets = np.asarray(sets, dtype=bool)
    if sets.ndim == 3 and sets.shape[-1] == 1:
        sets = sets[:, :, 0]
    if sets.shape != (len(probabilities), 12):
        raise ValueError(f'Unexpected MAPIE set shape {sets.shape}')
    return sets


In [ ]:
def adaptive_decision(capture_sets, tau=1.0, budget=3):
    sets = np.asarray(capture_sets, dtype=bool)
    if sets.ndim != 2 or sets.shape[1] != 12 or (not len(sets)) or (not 1 <= budget <= 3) or (tau <= 0):
        raise ValueError('Invalid sets, threshold, or budget')
    scores = np.zeros(12, dtype=np.float64)
    for used, mask in enumerate(sets[:budget], 1):
        size = int(mask.sum())
        if size:
            scores[mask] += 1.0 / size
        accepted = bool(scores.max() >= tau)
        if accepted or used == budget:
            tied = np.flatnonzero(scores == scores.max())
            return dict(prediction=int(tied[0]), captures=used, reinputs=used - 1, accepted=accepted, forced=not accepted, tied=len(tied) > 1, scores=scores.tolist())
    raise ValueError('Episode ended before acceptance or budget')


In [ ]:
def episode_layout(rows):
    cached = rows.attrs.get('episode_layout')
    if cached is not None:
        return cached
    names = []
    indices = []
    first = []
    for episode_id, g in rows.groupby('episode_id', sort=False):
        g = g.assign(capture_index=g.capture_index.astype(int)).sort_values('capture_index')
        names.append(episode_id)
        indices.append(g.index.to_numpy(dtype=int))
        first.append(int(g.index[0]))
    layout = np.full((len(names), 3), -1, dtype=int)
    lengths = np.asarray([len(ix) for ix in indices], dtype=int)
    for i, ix in enumerate(indices):
        layout[i, :len(ix)] = ix
    cached = (names, layout, lengths, rows.loc[first].copy())
    rows.attrs['episode_layout'] = cached
    return cached

def episode_metrics(rows, probabilities, sets, tau, budget, details=True):
    if rows.empty:
        raise ValueError('No episodes')
    names, layout, lengths, first = episode_layout(rows)
    n = len(names)
    scores = np.zeros((n, 12), dtype=np.float64)
    used = np.zeros(n, dtype=int)
    pred = np.full(n, -1, dtype=int)
    accepted = np.zeros(n, dtype=bool)
    tied = np.zeros(n, dtype=bool)
    active = np.ones(n, dtype=bool)
    for k in range(budget):
        if np.any(active & (lengths <= k)):
            raise ValueError('Episode ended before acceptance or budget')
        ix = np.flatnonzero(active)
        if not len(ix):
            break
        masks = sets[layout[ix, k]]
        sizes = masks.sum(axis=1)
        increments = np.divide(masks, sizes[:, None], out=np.zeros_like(masks, dtype=np.float64), where=sizes[:, None] > 0)
        scores[ix] += increments
        used[ix] = k + 1
        threshold = scores[ix].max(axis=1) >= tau
        finish = threshold | (k + 1 == budget)
        done = ix[finish]
        accepted[done] = threshold[finish]
        pred[done] = scores[done].argmax(axis=1)
        tied[done] = (scores[done] == scores[done].max(axis=1, keepdims=True)).sum(axis=1) > 1
        active[done] = False
    if active.any():
        raise RuntimeError('Incomplete adaptive decision')
    truth = first.button.to_numpy(dtype=int)
    initial = probabilities[layout[:, 0]].argmax(axis=1)
    processed = np.arange(3)[None, :] < used[:, None]
    capture_indices = layout[processed]
    capture_labels = rows.loc[capture_indices, 'button'].to_numpy(dtype=int)
    processed_sets = sets[capture_indices]
    set_sizes = processed_sets.sum(axis=1)
    covered = processed_sets[np.arange(len(capture_indices)), capture_labels]
    metrics = dict(episodes=int(n), processed_captures=int(len(capture_indices)), adaptive_accuracy=float(np.mean(pred == truth)), first_capture_accuracy=float(np.mean(initial == truth)), coverage=float(np.mean(covered)), mean_prediction_set_size=float(np.mean(set_sizes)), mean_reinputs=float(np.mean(used - 1)), mean_captures=float(np.mean(used)), acceptance_rate=float(np.mean(accepted)), forced_decision_rate=float(np.mean(~accepted)), tie_rate=float(np.mean(tied)), budget_reached_rate=float(np.mean(used == budget)))
    if not details:
        return (metrics, None, None)
    episodes = first.copy()
    episodes.attrs.clear()
    episodes['episode_id'] = names
    episodes['true_button'] = truth
    episodes['first_prediction'] = initial
    episodes['prediction'] = pred
    episodes['correct'] = pred == truth
    episodes['captures_used'] = used
    episodes['reinputs'] = used - 1
    episodes['accepted'] = accepted
    episodes['forced'] = ~accepted
    episodes['tied'] = tied
    episodes['evidence'] = [json.dumps(value.tolist()) for value in scores]
    episodes = episodes.reset_index(drop=True)
    captures = rows.loc[capture_indices].copy()
    captures.attrs.clear()
    captures['row_index'] = capture_indices
    captures['prediction_set_size'] = set_sizes
    captures['covered'] = covered
    captures['prediction_set'] = [json.dumps(np.flatnonzero(mask).tolist()) for mask in processed_sets]
    captures['model_prediction'] = probabilities[capture_indices].argmax(axis=1)
    captures['model_confidence'] = probabilities[capture_indices].max(axis=1)
    captures = captures.reset_index(drop=True)
    return (metrics, episodes, captures)


In [ ]:
def validation_report(name, run_id, predict, training_hashes, samples, run_output):
    records = []
    calibrators = {}
    cache = {}
    for domain, (raw, rows) in samples.items():
        if set(rows.response_sha256) & training_hashes:
            raise ValueError(f'{name}/{run_id}/{domain}: development overlap')
        cal = rows.index[rows.partition.eq('calibration')].to_numpy()
        val = rows.index[rows.partition.eq('validation')].to_numpy()
        p = np.full((len(rows), 12), np.nan, dtype=np.float64)
        p[cal] = predict(raw[cal])
        p[val] = predict(raw[val])
        labels = rows.button.to_numpy(dtype=int)
        validation_rows = rows.loc[val].copy()
        for alpha in RISK_LEVELS:
            cp = calibrate(p[cal], labels[cal], float(alpha))
            masks = np.zeros((len(rows), 12), dtype=bool)
            masks[val] = get_sets(cp, p[val])
            calibrators[domain, float(alpha)] = cp
            for tau in THRESHOLDS:
                for budget in CAPTURE_BUDGETS:
                    m, _, _ = episode_metrics(validation_rows, p, masks, float(tau), int(budget), details=False)
                    records.append(dict(model=name, run_id=run_id, domain=domain, alpha=float(alpha), tau=float(tau), max_captures=int(budget), **m))
        cache[domain] = p
        rows.to_csv(run_output / f'{domain}_capture_manifest.csv', index=False)
    grid = pd.DataFrame(records)
    grid.to_csv(run_output / 'validation_grid.csv', index=False)
    fixed = grid[np.isclose(grid.alpha, ALPHA[name]) & np.isclose(grid.tau, TAU) & grid.max_captures.eq(MAX_CAPTURES)]
    fixed.to_csv(run_output / 'validation_at_fixed_point.csv', index=False)
    macro = grid.groupby(['alpha', 'tau', 'max_captures'], as_index=False)[['adaptive_accuracy', 'mean_reinputs', 'mean_prediction_set_size', 'coverage']].mean()
    chosen = macro.sort_values(['adaptive_accuracy', 'mean_reinputs', 'mean_prediction_set_size', 'alpha', 'tau', 'max_captures'], ascending=[False, True, True, True, True, True]).iloc[0]
    write_json(run_output / 'reference_validation_selection.json', chosen.to_dict())
    if SELECTION_MODE == 'validation':
        selected_alpha, selected_tau, selected_budget = (float(chosen.alpha), float(chosen.tau), int(chosen.max_captures))
    else:
        selected_alpha, selected_tau, selected_budget = (float(ALPHA[name]), float(TAU), int(MAX_CAPTURES))
    frozen = dict(model=name, run_id=run_id, alpha=selected_alpha, tau=selected_tau, max_captures=selected_budget, selection=SELECTION_MODE, score=SCORE, include_last_label=INCLUDE_LAST_LABEL, tie_rule=TIE_RULE, cp_seed=CP_SEED)
    write_json(run_output / 'frozen_operating_point.json', frozen)
    return (calibrators, cache)


In [ ]:
def final_report(name, run_id, predict, samples, calibrators, cache, run_output):
    frozen = json.loads((run_output / 'frozen_operating_point.json').read_text())
    alpha, tau, budget = (float(frozen['alpha']), float(frozen['tau']), int(frozen['max_captures']))
    reports = []
    for domain, (raw, rows) in samples.items():
        final_ix = rows.index[rows.partition.eq('final')].to_numpy()
        p = cache[domain]
        p[final_ix] = predict(raw[final_ix])
        cp = calibrators[domain, alpha]
        sets = np.zeros((len(rows), 12), dtype=bool)
        sets[final_ix] = get_sets(cp, p[final_ix])
        metrics, episodes, captures = episode_metrics(rows.loc[final_ix], p, sets, tau, budget)
        reports.append(dict(model=name, run_id=run_id, domain=domain, alpha=alpha, tau=tau, max_captures=budget, **metrics))
        episodes.to_csv(run_output / f'{domain}_episode_decisions.csv', index=False)
        captures.to_csv(run_output / f'{domain}_processed_captures.csv', index=False)
        np.savez_compressed(run_output / f'{domain}_final_predictions.npz', row_index=final_ix, probabilities=p[final_ix], prediction_sets=sets[final_ix], labels=rows.loc[final_ix, 'button'].to_numpy(int))
        joblib.dump(cp, run_output / f'{domain}_calibrator.joblib')
        scores = getattr(getattr(cp, '_mapie_classifier', None), 'conformity_scores_', None)
        if scores is not None:
            np.save(run_output / f'{domain}_calibration_scores.npy', np.asarray(scores))
    table = pd.DataFrame(reports)
    table.to_csv(run_output / 'final_domain_metrics.csv', index=False)
    return table

def summarize(all_metrics, destination):
    all_metrics.to_csv(destination / 'all_run_domain_metrics.csv', index=False)
    cols = ['adaptive_accuracy', 'first_capture_accuracy', 'coverage', 'mean_prediction_set_size', 'mean_reinputs']
    summary = all_metrics.groupby(['model', 'domain'])[cols].agg(['mean', 'std', 'count'])
    summary.columns = ['_'.join(v) for v in summary.columns]
    summary.to_csv(destination / 'across_run_summary.csv')
    display(summary)
    np.savez_compressed(destination / 'CP_Final_Summary.npz', models=summary.index.get_level_values('model').to_numpy(dtype=str), domains=summary.index.get_level_values('domain').to_numpy(dtype=str), metric_names=summary.columns.to_numpy(dtype=str), values=summary.to_numpy(dtype=float))
    macro = all_metrics.groupby(['model', 'run_id'], as_index=False)[cols].mean()
    macro.to_csv(destination / 'per_run_macro_metrics.csv', index=False)
    weighted = []
    for (name, run_id), g in all_metrics.groupby(['model', 'run_id']):
        row = dict(model=name, run_id=run_id)
        for key in ('adaptive_accuracy', 'first_capture_accuracy', 'mean_reinputs'):
            row[key] = float(np.average(g[key], weights=g.episodes))
        for key in ('coverage', 'mean_prediction_set_size'):
            row[key] = float(np.average(g[key], weights=g.processed_captures))
        weighted.append(row)
    pd.DataFrame(weighted).to_csv(destination / 'per_run_response_episode_weighted_metrics.csv', index=False)


In [ ]:
# def plot_risk_curves(destination, names=None, metrics_to_plot=None):
#     import matplotlib.pyplot as plt
#     from matplotlib.ticker import FormatStrFormatter
#     plt.rcParams.update({'font.size': 8.5, 'pdf.fonttype': 42, 'ps.fonttype': 42})
#     colors = {'CC': '#D62728', 'CR': '#008000', 'CCR': '#003FBE'}
#     metrics = {'coverage': ('empirical_coverage', 'Empirical coverage (%)', 100), 'mean_prediction_set_size': ('average_prediction_set_size', 'Prediction set size', 1), 'adaptive_accuracy': ('adaptive_accuracy', 'Recognition accuracy (%)', 100), 'mean_reinputs': ('average_reinputs', 'Button re-inputs', 1)}
#     plot_dir = destination / ('validation_figures_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ_') + uuid.uuid4().hex[:8])
#     plot_dir.mkdir(exist_ok=False)
#     for name in MODELS if names is None else names:
#         grids = [pd.read_csv(destination / name / run_id / 'validation_grid.csv') for run_id in RUN_IDS]
#         grid = pd.concat(grids, ignore_index=True)
#         grid = grid[np.isclose(grid.tau, TAU) & grid.max_captures.eq(MAX_CAPTURES)]
#         for metric, (stem, ylabel, scale) in metrics.items():
#             if metrics_to_plot is not None and metric not in metrics_to_plot:
#                 continue
#             summary = grid.groupby(['domain', 'alpha'])[metric].agg(['mean', 'std', 'count']).reset_index()
#             summary.to_csv(plot_dir / f'{name}_{stem}_validation_values.csv', index=False)
#             display(summary)
#             np.savez_compressed(plot_dir / f'{name}_{stem}.npz', domains=summary.domain.to_numpy(dtype=str), risk_levels=summary.alpha.to_numpy(), mean=summary['mean'].to_numpy() * scale, sample_sd=summary['std'].to_numpy() * scale, count=summary['count'].to_numpy(), metric=np.asarray(metric), scale=np.asarray(scale))
#             fig, ax = plt.subplots(figsize=(3.8, 2.8), layout='constrained')
#             for domain in TEST:
#                 g = summary[summary.domain.eq(domain)].sort_values('alpha')
#                 x = g.alpha.to_numpy()
#                 mean = g['mean'].to_numpy() * scale
#                 ax.plot(x, mean, color=colors[domain], linewidth=1.5, label=domain)
#                 if (g['count'] > 1).all():
#                     sd = g['std'].to_numpy() * scale
#                     ax.fill_between(x, mean - sd, mean + sd, color=colors[domain], alpha=0.18, linewidth=0)
#             ax.set(xlabel='Risk level ($\\alpha$)', ylabel=ylabel, xlim=(0.05, 0.3))
#             ax.set_xticks(np.arange(0.05, 0.301, 0.05))
#             ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
#             ax.tick_params(direction='in', top=True, right=True)
#             ax.grid(alpha=0.3, linestyle=':')
#             ax.legend(fontsize=8.5)
#             fig.savefig(plot_dir / f'{name}_{stem}_validation.pdf')
#             fig.savefig(plot_dir / f'{name}_{stem}_validation.png', dpi=600)
#             plt.show()
#             plt.close(fig)


In [ ]:
class AdaptivePINRecognizer:

    def __init__(self, name, run_id, domain, cp_output=None):
        if cp_output is None:
            raise ValueError('Set cp_output to a completed CP execution directory')
        directory = Path(cp_output) / name / run_id
        self.predict, model_reference = load_predictor(name, run_id)
        expected = json.loads((directory / 'model_reference.json').read_text())
        if model_reference != expected:
            raise ValueError('Checkpoint or preprocessing mismatch')
        self.settings = json.loads((directory / 'frozen_operating_point.json').read_text())
        if self.settings['score'] != SCORE or self.settings['include_last_label'] != INCLUDE_LAST_LABEL:
            raise ValueError('Score configuration mismatch')
        self.cp = joblib.load(directory / f'{domain}_calibrator.joblib')
        self.reset()

    def reset(self):
        self.evidence = np.zeros(12, dtype=np.float64)
        self.captures = 0

    def capture(self, complex_response):
        raw = np.asarray(complex_response)
        if raw.ndim != 1:
            raise ValueError('Expected one response')
        mask = get_sets(self.cp, self.predict(raw[None, :]))[0]
        size = int(mask.sum())
        if size:
            self.evidence[mask] += 1.0 / size
        self.captures += 1
        accepted = bool(self.evidence.max() >= self.settings['tau'])
        finished = accepted or self.captures >= self.settings['max_captures']
        result = dict(action='accept' if accepted else 'budget_fallback' if finished else 'request_reinput', prediction=int(self.evidence.argmax()) if finished else None, captures=self.captures, reinputs=self.captures - 1, prediction_set=np.flatnonzero(mask).tolist(), evidence=self.evidence.tolist(), accepted=accepted)
        if finished:
            self.reset()
        return result


In [ ]:
SHARED_SAMPLES = None
SHARED_DEVELOPMENT_HASHES = None

def prepare_cp_execution(name):
    global SHARED_SAMPLES, SHARED_DEVELOPMENT_HASHES
    if importlib.metadata.version('MAPIE') != '1.6.0':
        raise RuntimeError('Install MAPIE==1.6.0')
    for run_id in RUN_IDS:
        model_files(name, run_id)
    if SHARED_SAMPLES is None:
        manifest = load_capture_manifest()
        SHARED_SAMPLES = load_test_responses(manifest)
    destination = OUTPUT / (name + datetime.now(timezone.utc).strftime('_cp_%Y%m%dT%H%M%S%fZ_') + uuid.uuid4().hex[:8])
    destination.mkdir(parents=True, exist_ok=False)
    if SHARED_DEVELOPMENT_HASHES is None:
        SHARED_DEVELOPMENT_HASHES = audit_development(SHARED_SAMPLES, destination)
    else:
        pd.DataFrame({'response_sha256': sorted(SHARED_DEVELOPMENT_HASHES)}).to_csv(destination / 'development_input_hashes.csv', index=False)
    manifest = pd.concat([rows for raw, rows in SHARED_SAMPLES.values()], ignore_index=True)
    manifest.to_csv(destination / 'capture_manifest.csv', index=False)
    versions = {p: importlib.metadata.version(p) for p in ('numpy', 'pandas', 'scikit-learn', 'MAPIE', 'joblib')}
    write_json(destination / 'experiment.json', dict(model=name, runs=list(RUN_IDS), versions=versions, python=platform.python_version(), manifest_sha256=file_sha256(CAPTURES_CSV), score=SCORE, include_last_label=INCLUDE_LAST_LABEL, cp_seed=CP_SEED, selection_mode=SELECTION_MODE, risk_levels=RISK_LEVELS.tolist(), thresholds=THRESHOLDS.tolist(), capture_budgets=list(CAPTURE_BUDGETS), preprocessing=PREPROCESSING))
    return (destination, SHARED_SAMPLES, SHARED_DEVELOPMENT_HASHES)


In [ ]:
RUN_DL_EVALUATION = False
DL_RUN_ROOT = None
DL_CP_OUTPUT = None
DL_METRICS = None


In [ ]:
if RUN_DL_EVALUATION:
    DL_CP_OUTPUT, DL_SAMPLES, DL_DEVELOPMENT_HASHES = prepare_cp_execution('DL')
    print('DL output:', DL_CP_OUTPUT)


In [ ]:
if RUN_DL_EVALUATION:
    DL_PREDICTORS = {}
    for run_id in RUN_IDS:
        predictor, metadata = load_predictor('DL', run_id)
        folder = DL_CP_OUTPUT / 'DL' / run_id
        folder.mkdir(parents=True, exist_ok=False)
        write_json(folder / 'model_reference.json', metadata)
        DL_PREDICTORS[run_id] = predictor
        print(run_id, metadata['model_path'])


In [ ]:
if RUN_DL_EVALUATION:
    DL_CALIBRATORS = {}
    DL_PROBABILITIES = {}
    for run_id in RUN_IDS:
        print('DL calibration and validation:', run_id, flush=True)
        calibrators, probabilities = validation_report('DL', run_id, DL_PREDICTORS[run_id], DL_DEVELOPMENT_HASHES, DL_SAMPLES, DL_CP_OUTPUT / 'DL' / run_id)
        DL_CALIBRATORS[run_id] = calibrators
        DL_PROBABILITIES[run_id] = probabilities
        display(pd.DataFrame([json.loads((DL_CP_OUTPUT / 'DL' / run_id / 'frozen_operating_point.json').read_text())]))


In [ ]:
if RUN_DL_EVALUATION:
    DL_REPORTS = []
    for run_id in RUN_IDS:
        print('DL final evaluation:', run_id, flush=True)
        report = final_report('DL', run_id, DL_PREDICTORS[run_id], DL_SAMPLES, DL_CALIBRATORS[run_id], DL_PROBABILITIES[run_id], DL_CP_OUTPUT / 'DL' / run_id)
        DL_REPORTS.append(report)
        display(report)
    DL_METRICS = pd.concat(DL_REPORTS, ignore_index=True)


In [ ]:
if RUN_DL_EVALUATION:
    DL_SUMMARY_OUTPUT = DL_CP_OUTPUT / 'summary'
    DL_SUMMARY_OUTPUT.mkdir(exist_ok=False)
    summarize(DL_METRICS, DL_SUMMARY_OUTPUT)


In [ ]:
# if RUN_DL_EVALUATION:
#     plot_risk_curves(DL_CP_OUTPUT, names=['DL'], metrics_to_plot=['coverage'])


In [ ]:
# if RUN_DL_EVALUATION:
#     plot_risk_curves(DL_CP_OUTPUT, names=['DL'], metrics_to_plot=['mean_prediction_set_size'])


In [ ]:
# if RUN_DL_EVALUATION:
#     plot_risk_curves(DL_CP_OUTPUT, names=['DL'], metrics_to_plot=['adaptive_accuracy'])


In [ ]:
# if RUN_DL_EVALUATION:
#     plot_risk_curves(DL_CP_OUTPUT, names=['DL'], metrics_to_plot=['mean_reinputs'])


In [ ]:
RUN_DL_LIVE_INFERENCE = False
DL_LIVE_RUN_ID = 'run1'
DL_LIVE_DOMAIN = 'CC'
DL_LIVE_CP_OUTPUT = None
DL_LIVE_INPUT_PATH = None
if RUN_DL_LIVE_INFERENCE:
    output = DL_LIVE_CP_OUTPUT if DL_LIVE_CP_OUTPUT is not None else DL_CP_OUTPUT
    if output is None or DL_LIVE_INPUT_PATH is None:
        raise ValueError('Set DL_LIVE_CP_OUTPUT and DL_LIVE_INPUT_PATH')
    recognizer = AdaptivePINRecognizer('DL', DL_LIVE_RUN_ID, DL_LIVE_DOMAIN, cp_output=output)
    responses = np.load(DL_LIVE_INPUT_PATH, allow_pickle=False)
    if responses.ndim != 2 or not 1 <= len(responses) <= MAX_CAPTURES:
        raise ValueError('Supply one ordered episode with up to three response rows')
    for response in responses:
        decision = recognizer.capture(response)
        print(json.dumps(decision, indent=2))
        if decision['action'] != 'request_reinput':
            break


In [ ]:
RUN_RF_EVALUATION = False
RF_RUN_ROOT = None
RF_CP_OUTPUT = None
RF_METRICS = None


In [ ]:
if RUN_RF_EVALUATION:
    RF_CP_OUTPUT, RF_SAMPLES, RF_DEVELOPMENT_HASHES = prepare_cp_execution('RF')
    print('RF output:', RF_CP_OUTPUT)


In [ ]:
if RUN_RF_EVALUATION:
    RF_PREDICTORS = {}
    for run_id in RUN_IDS:
        predictor, metadata = load_predictor('RF', run_id)
        folder = RF_CP_OUTPUT / 'RF' / run_id
        folder.mkdir(parents=True, exist_ok=False)
        write_json(folder / 'model_reference.json', metadata)
        RF_PREDICTORS[run_id] = predictor
        print(run_id, metadata['model_path'])


In [ ]:
if RUN_RF_EVALUATION:
    RF_CALIBRATORS = {}
    RF_PROBABILITIES = {}
    for run_id in RUN_IDS:
        print('RF calibration and validation:', run_id, flush=True)
        calibrators, probabilities = validation_report('RF', run_id, RF_PREDICTORS[run_id], RF_DEVELOPMENT_HASHES, RF_SAMPLES, RF_CP_OUTPUT / 'RF' / run_id)
        RF_CALIBRATORS[run_id] = calibrators
        RF_PROBABILITIES[run_id] = probabilities
        display(pd.DataFrame([json.loads((RF_CP_OUTPUT / 'RF' / run_id / 'frozen_operating_point.json').read_text())]))


In [ ]:
if RUN_RF_EVALUATION:
    RF_REPORTS = []
    for run_id in RUN_IDS:
        print('RF final evaluation:', run_id, flush=True)
        report = final_report('RF', run_id, RF_PREDICTORS[run_id], RF_SAMPLES, RF_CALIBRATORS[run_id], RF_PROBABILITIES[run_id], RF_CP_OUTPUT / 'RF' / run_id)
        RF_REPORTS.append(report)
        display(report)
    RF_METRICS = pd.concat(RF_REPORTS, ignore_index=True)


In [ ]:
if RUN_RF_EVALUATION:
    RF_SUMMARY_OUTPUT = RF_CP_OUTPUT / 'summary'
    RF_SUMMARY_OUTPUT.mkdir(exist_ok=False)
    summarize(RF_METRICS, RF_SUMMARY_OUTPUT)


In [ ]:
# if RUN_RF_EVALUATION:
#     plot_risk_curves(RF_CP_OUTPUT, names=['RF'], metrics_to_plot=['coverage'])


In [ ]:
# if RUN_RF_EVALUATION:
#     plot_risk_curves(RF_CP_OUTPUT, names=['RF'], metrics_to_plot=['mean_prediction_set_size'])


In [ ]:
# if RUN_RF_EVALUATION:
#     plot_risk_curves(RF_CP_OUTPUT, names=['RF'], metrics_to_plot=['adaptive_accuracy'])


In [ ]:
# if RUN_RF_EVALUATION:
#     plot_risk_curves(RF_CP_OUTPUT, names=['RF'], metrics_to_plot=['mean_reinputs'])


In [ ]:
RUN_RF_LIVE_INFERENCE = False
RF_LIVE_RUN_ID = 'run1'
RF_LIVE_DOMAIN = 'CC'
RF_LIVE_CP_OUTPUT = None
RF_LIVE_INPUT_PATH = None
if RUN_RF_LIVE_INFERENCE:
    output = RF_LIVE_CP_OUTPUT if RF_LIVE_CP_OUTPUT is not None else RF_CP_OUTPUT
    if output is None or RF_LIVE_INPUT_PATH is None:
        raise ValueError('Set RF_LIVE_CP_OUTPUT and RF_LIVE_INPUT_PATH')
    recognizer = AdaptivePINRecognizer('RF', RF_LIVE_RUN_ID, RF_LIVE_DOMAIN, cp_output=output)
    responses = np.load(RF_LIVE_INPUT_PATH, allow_pickle=False)
    if responses.ndim != 2 or not 1 <= len(responses) <= MAX_CAPTURES:
        raise ValueError('Supply one ordered episode with up to three response rows')
    for response in responses:
        decision = recognizer.capture(response)
        print(json.dumps(decision, indent=2))
        if decision['action'] != 'request_reinput':
            break


In [ ]:
# RUN_ARCHIVED_PLOTS = False
# ARCHIVED_CP_RUN = None
# ARCHIVED_MODEL = 'DL'
# if RUN_ARCHIVED_PLOTS:
#     if ARCHIVED_CP_RUN is None or ARCHIVED_MODEL not in MODELS:
#         raise ValueError('Set ARCHIVED_CP_RUN and select DL or RF')
#     plot_risk_curves(Path(ARCHIVED_CP_RUN), names=[ARCHIVED_MODEL])
